# Semantic Gaussians: Bonsai fusion on Colab GPU

Run every cell in order on a fresh **GPU** Colab server. The notebook stops early when CUDA is unavailable, uses the pinned `sega` environment, stores the Bonsai dataset and checkpoint permanently in Google Drive, and copies final features there too.

In [ ]:
# 1. Refuse to continue on a CPU server.
import shutil
import subprocess
import torch

assert shutil.which("nvidia-smi"), (
    "No NVIDIA GPU was found. Select Kernel > Colab > "
    "New Colab Server > T4 GPU, then rerun this cell."
)
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "The notebook Python cannot access CUDA."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 2. Mount persistent storage and define all paths once.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
REPO = Path("/content/semantic-gaussians")
CONDA = Path("/content/miniforge3/bin/conda")
DRIVE_ROOT = Path("/content/drive/MyDrive/semantic-gaussians")
DRIVE_DATASET = DRIVE_ROOT / "data/bonsai"
DRIVE_CHECKPOINT = DRIVE_ROOT / "bonsai/checkpoint"
DRIVE_OUTPUT = DRIVE_ROOT / "outputs/bonsai/fused"
for path in (DRIVE_DATASET, DRIVE_CHECKPOINT, DRIVE_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
print("Persistent dataset:", DRIVE_DATASET)
print("Persistent checkpoint:", DRIVE_CHECKPOINT)
print("Persistent output:", DRIVE_OUTPUT)

In [ ]:
# 3. Clone once and initialize the CUDA submodules.
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--recursive",
         "https://github.com/sharinka0715/semantic-gaussians.git", str(REPO)],
        check=True,
    )
else:
    print("Repository already exists; skipping clone.")
subprocess.run(
    ["git", "submodule", "update", "--init", "--recursive"],
    cwd=REPO, check=True,
)

In [ ]:
# 4. Install Miniforge and create the repository's pinned environment.
import json

if not CONDA.exists():
    installer = Path("/tmp/miniforge.sh")
    subprocess.run(
        ["wget", "-q",
         "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
         "-O", str(installer)], check=True,
    )
    subprocess.run(
        ["bash", str(installer), "-b", "-p", "/content/miniforge3"],
        check=True,
    )
else:
    print("Miniforge already exists; skipping installation.")

envs = json.loads(subprocess.run(
    [str(CONDA), "env", "list", "--json"],
    check=True, text=True, capture_output=True,
).stdout)["envs"]
if not any(Path(path).name == "sega" for path in envs):
    subprocess.run(
        [str(CONDA), "env", "create", "--file", str(REPO / "environment.yml")],
        check=True,
    )
else:
    print("Environment 'sega' already exists; skipping creation.")

In [ ]:
# 5. Install packages and compile the custom CUDA extensions once per server.
marker = Path("/content/.semantic_gaussians_requirements_ready")
if not marker.exists():
    subprocess.run(
        [str(CONDA), "run", "--no-capture-output", "-n", "sega",
         "pip", "install", "-r", "requirements.txt"],
        cwd=REPO, check=True,
    )
    subprocess.run(
        [str(CONDA), "run", "--no-capture-output", "-n", "sega",
         "pip", "install", "huggingface_hub[cli]", "gdown"],
        check=True,
    )
    marker.touch()
else:
    print("Requirements already installed; skipping.")

In [ ]:
# 6. Verify CUDA in the exact Python environment used below.
test_code = """
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable inside sega'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
import rgbd_rasterization
import channel_rasterization
from simple_knn._C import distCUDA2
print('All custom CUDA extensions imported successfully.')
"""
subprocess.run(
    [str(CONDA), "run", "--no-capture-output", "-n", "sega",
     "python", "-c", test_code],
    cwd=REPO, check=True,
)

## Download and verify the Bonsai inputs

This workflow uses the repository's recommended 4× Mip-NeRF 360 resolution (`779×519`) to reduce Colab memory use.

In [ ]:
# 7. Download 292 photographs and COLMAP data to Drive (first run only).
hf = Path("/content/miniforge3/envs/sega/bin/hf")
dataset_root = DRIVE_DATASET
for include in ("bonsai/images_4/*", "bonsai/sparse/0/*", "bonsai/nb-info.json"):
    subprocess.run(
        [str(hf), "download", "rishitdagli/nerf-gs-datasets",
         "--repo-type", "dataset", "--include", include,
         "--local-dir", str(DRIVE_ROOT / "data")],
        check=True,
    )
images = list((dataset_root / "images_4").glob("*.JPG"))
sparse = [dataset_root / "sparse/0" / name for name in
          ("cameras.bin", "images.bin", "points3D.bin")]
assert len(images) == 292, f"Expected 292 images, found {len(images)}"
assert all(path.is_file() for path in sparse), "COLMAP files are incomplete"
print("Dataset verified:", dataset_root)

In [ ]:
# 8. Download the pretrained checkpoint to Drive when it is not cached there.
checkpoint_ply = DRIVE_CHECKPOINT / "point_cloud/iteration_30000/point_cloud.ply"
if not checkpoint_ply.is_file():
    archive = Path("/content/bonsai-checkpoint.zip")
    subprocess.run(
        ["wget", "-c",
         "https://data.ciirc.cvut.cz/public/projects/2023NerfBaselines/data/gaussian-splatting/mipnerf360/bonsai.zip",
         "-O", str(archive)], check=True,
    )
    (DRIVE_ROOT / "bonsai").mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["unzip", "-q", "-o", str(archive), "-d", str(DRIVE_ROOT / "bonsai")],
        check=True,
    )
assert checkpoint_ply.is_file(), f"Missing checkpoint: {checkpoint_ply}"
print("Checkpoint verified:", checkpoint_ply)

In [ ]:
# 9. Download and locate the OpenSeg SavedModel required by fusion.py.
openseg_model = REPO / "weights/openseg_exported_clip"
saved_model = openseg_model / "saved_model.pb"
if not saved_model.is_file():
    archive = Path("/content/openseg-model.zip")
    unpacked = Path("/content/openseg-unpacked")
    subprocess.run(
        [str(CONDA), "run", "--no-capture-output", "-n", "sega",
         "gdown", "--fuzzy",
         "https://drive.google.com/file/d/1DgyH-1124Mo8p6IUJ-ikAiwVZDDfteak/view",
         "-O", str(archive)], check=True,
    )
    unpacked.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", str(archive), "-d", str(unpacked)], check=True)
    candidates = list(unpacked.rglob("saved_model.pb"))
    assert candidates, "The OpenSeg archive did not contain saved_model.pb"
    shutil.copytree(candidates[0].parent, openseg_model, dirs_exist_ok=True)
assert saved_model.is_file(), f"OpenSeg model is incomplete: {saved_model}"
print("OpenSeg model verified:", openseg_model)

In [ ]:
# 10. Run fusion through sega, not the notebook's base Python.
local_fused = REPO / "bonsai/fused"
local_fused.mkdir(parents=True, exist_ok=True)
command = [
    str(CONDA), "run", "--no-capture-output", "-n", "sega",
    "python", "fusion.py",
    f"scene.scene_path={DRIVE_DATASET}",
    "scene.colmap_images=images_4",
    f"model.model_dir={DRIVE_CHECKPOINT}",
    "model.load_iteration=30000",
    "fusion.img_dim=[779,519]",
    "fusion.num_workers=2",
    "fusion.out_dir=./bonsai/fused",
]
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
# 11. Copy generated features to Drive before removing the Colab server.
fused_files = list((REPO / "bonsai/fused").glob("*.pt"))
assert fused_files, "Fusion did not produce any .pt files"
for source in fused_files:
    shutil.copy2(source, DRIVE_OUTPUT / source.name)
print(f"Copied {len(fused_files)} file(s) to {DRIVE_OUTPUT}")

## Important limitations

OpenSeg creates a 768-dimensional semantic component for every Gaussian. Even with 4× images, the pretrained Bonsai model can approach a free T4's memory limit. If the final cell reports CUDA out-of-memory, stop it before trying another configuration. The Bonsai dataset, checkpoint, and fused output persist in Drive; the Conda environment, compiled packages, repository clone, and OpenSeg download remain temporary and must be recreated on a new server.